# 03 - chain 1 samples and base model

Makes `c1_train_S.npy` (first 1M rows of the learning-curve training part) and the fixed 500k test samples of weeks 23-30 (seed 0), trains the chain 1 base model and checks it matches the learning curve. The KEEP rows here use the older run name `1a`; the final KEEP results come from `run_grid.py`. Paths point to our shared Drive folder.

In [1]:
import os, sys, gc
from pathlib import Path
PROJECT = Path(r"G:\My Drive\CESNET_Project")
os.environ["CESNET_PROJECT"] = str(PROJECT)
sys.path.insert(0, str(PROJECT / "code"))
import numpy as np, pandas as pd
import common as C

# S sample = first 1M of the learning-curve training part (validation rows never used)
S = C.load_json("params.json")["S"]; assert S == 1_000_000
train_perm = np.load(C.SAMPLES_DIR / "lc_w18_train_perm.npy")
s_idx = np.sort(train_perm[:S])
np.save(C.SAMPLES_DIR / "c1_train_S.npy", s_idx)
print(f"S sample: {len(s_idx):,} rows of week 18")

# fixed test sample per test week: 500k random 10-class flows, seed 0, made once
rows = []
for w in range(23, 31):
    path = C.SAMPLES_DIR / f"c1_test_w{w}.npy"
    df = C.load_week(w)
    n = len(df)
    if path.exists():
        idx = np.load(path)                       # never redraw an existing sample
    else:
        rng = np.random.default_rng(0)
        idx = np.sort(rng.choice(n, size=min(500_000, n), replace=False))
        np.save(path, idx)
    cnt = np.bincount(df["y"].to_numpy()[idx], minlength=10)
    rows.append({"week": w, "flows_10class": n, "test_rows": len(idx),
                 **{f"class_{i}": int(cnt[i]) for i in range(10)}})
    print(f"week {w}: {n:>9,} 10-class flows -> test {len(idx):,}, smallest class {cnt.min():,}")
    del df; gc.collect()

support = pd.DataFrame(rows)
support.to_csv(C.RESULTS_DIR / "c1_test_support.csv", index=False)
C.log_event("A4.1: c1_train_S.npy (first 1M of lc_w18_train_perm, sorted) and "
            "c1_test_w23..w30.npy (500k per week, seed 0, sorted) created", who="Aya")
support

S sample: 1,000,000 rows of week 18
week 23: 3,230,929 10-class flows -> test 500,000, smallest class 11,475
week 24: 3,179,219 10-class flows -> test 500,000, smallest class 18,474
week 25: 3,398,815 10-class flows -> test 500,000, smallest class 25,907
week 26: 2,481,248 10-class flows -> test 500,000, smallest class 26,792
week 27: 1,736,891 10-class flows -> test 500,000, smallest class 25,085
week 28: 1,842,344 10-class flows -> test 500,000, smallest class 10,223
week 29: 1,698,620 10-class flows -> test 500,000, smallest class 10,788
week 30: 1,649,003 10-class flows -> test 500,000, smallest class 12,936


,week,flows_10class,test_rows,class_0,class_1,class_2,class_3,class_4,class_5,class_6,class_7,class_8,class_9
0,23,3230929,500000,104231,57606,52461,35155,55700,50556,41781,52400,11475,38635
1,24,3179219,500000,103628,60090,49873,33185,53435,47182,42604,53205,18474,38324
2,25,3398815,500000,105642,52710,50045,33388,50446,52146,38475,53082,25907,38159
3,26,2481248,500000,103691,56881,50742,32197,48988,44593,40596,61344,26792,34176
4,27,1736891,500000,119598,43482,59287,36235,45942,35949,34609,65187,25085,34626
5,28,1842344,500000,103026,67720,49962,44383,61565,10223,46502,57521,29825,29273
6,29,1698620,500000,103380,69159,49840,49218,60653,10788,48873,47380,30618,30091
7,30,1649003,500000,105735,70382,50366,51172,52803,12936,50434,44881,31032,30259


In [2]:
import time, lightgbm as lgb

# ---- 1. base model on S ----
df18 = C.load_week(18)
X18, y18 = C.to_xy(df18); b18 = df18["bytes_total"].to_numpy(); del df18; gc.collect()
s_idx = np.load(C.SAMPLES_DIR / "c1_train_S.npy")
val_idx = np.load(C.SAMPLES_DIR / "lc_w18_val_idx.npy")
assert len(np.intersect1d(s_idx, val_idx)) == 0, "S overlaps validation!"
Xv, yv, bv = X18[val_idx], y18[val_idx], b18[val_idx]
Xs, ys = X18[s_idx], y18[s_idx]
del X18, y18, b18; gc.collect()

C.MODELS_DIR.mkdir(parents=True, exist_ok=True)
model_path = C.MODELS_DIR / "c1_base.txt"
with C.Measure() as m:
    booster = lgb.train(C.lgb_params(), lgb.Dataset(Xs, label=ys),
                        num_boost_round=C.rounds()["base"])
booster.save_model(str(model_path)); C.sync_to_drive(model_path)
del Xs, ys; gc.collect()

# ---- 2. reproducibility check against the learning curve (1M point = 0.839876) ----
sv, _, _ = C.evaluate(yv, C.predict_classes(booster, Xv), bv)
print(f"base model: {m.seconds:.1f} s, peak {m.peak_mb/1024:.1f} GB, "
      f"{booster.num_trees()} trees, {C.model_kb(model_path):.0f} KB")
print(f"validation macro-F1 {sv['macro_f1']:.6f} (learning curve 1M: 0.839876)")
C.log_event(f"A4.2: c1_base trained on c1_train_S ({m.seconds:.1f} s, peak {m.peak_mb:.0f} MB); "
            f"val macro-F1 {sv['macro_f1']:.6f}", who="Aya")

# ---- 3. KEEP on test weeks 23-30 ----
support = pd.read_csv(C.RESULTS_DIR / "c1_test_support.csv").set_index("week")
run_id = C.make_run_id(1, "1a", "KEEP", 0, "none", 0)
done = C.done_keys(1)
for w in range(23, 31):
    if (run_id, w) in done:
        print(f"week {w}: already in results_c1.csv, skipping"); continue
    df = C.load_week(w)
    idx = np.load(C.SAMPLES_DIR / f"c1_test_w{w}.npy")
    X, y = C.to_xy(df.iloc[idx]); b = df["bytes_total"].to_numpy()[idx]
    del df; gc.collect()
    pred, psec, fps = C.predict_timed(booster, X)
    s, per_class, cm = C.evaluate(y, pred, b,
                                  week_scale=support.loc[w, "flows_10class"] / len(idx))
    C.append_result({"run_id": run_id, "chain": 1, "tier": "1a", "strategy": "KEEP",
                     "budget": 0, "sampling": "none", "seed": 0, "label_week": 0,
                     "n_queried": 0, "n_used": 0, "train_rows": len(s_idx),
                     "n_trees": booster.num_trees(), "test_week": w, **s,
                     "update_time_s": 0.0, "peak_mem_mb": 0.0,
                     "model_kb": C.model_kb(model_path),
                     "predict_time_s": psec, "flows_per_s": fps}, per_class, cm)
    print(f"week {w}: KEEP macro-F1 {s['macro_f1']:.4f}  acc {s['accuracy']:.4f}  "
          f"byte_err {s['byte_err']:.3f}")

res = pd.read_csv(C.results_path(1))
res[res["strategy"] == "KEEP"][["test_week", "macro_f1", "accuracy", "byte_err",
                                "misclassified_gb_week_est"]]

base model: 38.9 s, peak 1.0 GB, 1000 trees, 6870 KB
validation macro-F1 0.839876 (learning curve 1M: 0.839876)
week 23: KEEP macro-F1 0.8089  acc 0.8163  byte_err 0.366
week 24: KEEP macro-F1 0.8145  acc 0.8180  byte_err 0.356
week 25: KEEP macro-F1 0.7930  acc 0.7951  byte_err 0.419
week 26: KEEP macro-F1 0.7703  acc 0.7793  byte_err 0.468
week 27: KEEP macro-F1 0.7730  acc 0.7838  byte_err 0.437
week 28: KEEP macro-F1 0.7356  acc 0.7605  byte_err 0.435
week 29: KEEP macro-F1 0.7335  acc 0.7509  byte_err 0.435
week 30: KEEP macro-F1 0.7382  acc 0.7490  byte_err 0.445


,test_week,macro_f1,accuracy,byte_err,misclassified_gb_week_est
0,23,0.808914,0.816300,0.366009,130.257826
1,24,0.814517,0.817968,0.355800,118.567622
2,25,0.792953,0.795136,0.419338,206.081274
3,26,0.770295,0.779320,0.468126,198.948133
4,27,0.773049,0.783808,0.437415,126.444312
5,28,0.735570,0.760486,0.434650,176.311203
6,29,0.733458,0.750874,0.435406,174.782302
7,30,0.738188,0.748986,0.445063,161.651601


In [3]:
import pandas as pd
proc = sorted(p.stem for p in (C.PROJECT_ROOT / "data" / "processed").glob("*.parquet"))
print("processed weeks:", proc)
print("budget sample files:", len(list(C.SAMPLES_DIR.glob("*budget*"))))
print("code files:", sorted(p.name for p in (C.PROJECT_ROOT / "code").glob("*.py")))
r = pd.read_csv(C.PROJECT_ROOT / "results" / "results_c1.csv")
print("\nrows in results_c1.csv by strategy:")
print(r.groupby("strategy")["run_id"].agg(["nunique", "size"]))

processed weeks: ['week_17', 'week_18', 'week_22', 'week_23', 'week_24', 'week_25', 'week_26', 'week_27', 'week_28', 'week_29', 'week_30', 'week_36', 'week_40', 'week_41', 'week_42', 'week_43']
budget sample files: 40
code files: ['common.py', 'download_weeks.py', 'make_samples_l4.py', 'preprocess.py', 'run_grid.py']

rows in results_c1.csv by strategy:
          nunique  size
strategy               
KEEP            1     8
